# Phase 13 — Data Snapshot, Versioning, and Contract Freezing

This notebook builds immutable data snapshots and freezes model-core schema contracts before new labels, pairs, features, or model training artifacts are generated.

Durable outputs are written under `reports/` so later model cards, baseline notebooks, and export manifests can reference the exact data and schema state used for training decisions.


## Step 13.1 — Source schema definitions

### Purpose
Define every source schema that may feed model-core training or evaluation before features and labels are generated.

### Required input
Legacy jobs, legacy profiles, legacy pair labels, future CV text, parsed CV sections, backend candidate job sets, manual validation labels, and the Backend API OpenAPI contract.

### Action
Create explicit notebook config cells for each source. Each schema records owner, schema version, identifier fields, required columns, optional columns, safe-use policy, and whether the source is available in the repository snapshot.

### Expected output
A versioned source schema registry that future notebooks can reuse when validating raw inputs and constructing frozen data snapshots.

### Verification
The registry must include jobs, profiles, CV text, parsed CV sections, candidate job sets, manual labels, and legacy weak-label pairs. Available repository files must exist before snapshot generation.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

PHASE_ID = "phase_13_data_snapshot_contract_freezing"
SEED = 202613
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training" / "notebooks").exists():
            return candidate
    raise RuntimeError("Repository root not found. Start notebook inside bisakerja-model repository.")


REPO_ROOT = find_repo_root()
REPORTS_DIR = REPO_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

SCHEMA_VERSION = "data-snapshot-contract-v1"
ALLOWED_LANGUAGES = {"ID", "EN", "MIXED", "UNKNOWN"}
SCORE_RANGE_0_100 = {"min": 0, "max": 100}
SCORE_RANGE_0_1 = {"min": 0.0, "max": 1.0}


def rel(path: Path) -> str:
    return path.resolve().relative_to(REPO_ROOT).as_posix()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [json_safe(v) for v in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value

source_schema_registry: dict[str, dict[str, Any]] = {
    "jobs_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training",
        "path": "legacy/dataset/indotech_job_cleaned.csv",
        "format": "csv",
        "available_in_repo": True,
        "primary_key": ["job_id"],
        "required_columns": ["job_id", "title", "description", "requirements_concat", "skills_clean", "experience_level", "language_signal", "status"],
        "optional_columns": ["company_name", "normalized_title", "category", "work_type", "employment_type", "province", "city", "salary_min", "salary_max", "salary_currency", "source_posted_at"],
        "language_column": "language_signal",
        "safe_training_use": "Use job text, normalized role, skills, requirements, experience level, language, and coarse location/work-type features. Do not train on company identity as a primary fit signal.",
    },
    "profiles_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training",
        "path": "legacy/dataset/techtalent_profile_cleaned.csv",
        "format": "csv",
        "available_in_repo": True,
        "primary_key": ["ID"],
        "required_columns": ["ID", "Skills", "Projects", "Education", "Experience", "Job_Role", "Required_Skills"],
        "optional_columns": [],
        "safe_training_use": "Use skills, projects, education band, experience band, target role, and required-skill text. Treat profile ID only as split/group key.",
    },
    "legacy_pairs_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training",
        "path": "legacy/artifacts/pairs.parquet",
        "format": "parquet",
        "available_in_repo": True,
        "primary_key": ["profile_id", "job_id"],
        "required_columns": ["profile_id", "job_id", "profile_text", "job_text", "profile_skills", "job_skills", "profile_exp", "job_exp", "fit_score"],
        "score_column": "fit_score",
        "score_range": SCORE_RANGE_0_1,
        "safe_training_use": "Prototype weak-label evidence only. Do not use as production training evidence after pairs_v2 is generated.",
    },
    "cv_text_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training",
        "path": None,
        "format": "table-or-jsonl",
        "available_in_repo": False,
        "primary_key": ["cv_id"],
        "required_columns": ["cv_id", "profile_id", "source_type", "raw_text", "parser_version", "language_signal", "created_at"],
        "unsafe_columns": ["email", "phone", "address", "raw_file_bytes", "storage_signed_url"],
        "safe_training_use": "Use parsed text and parser metadata only after PII removal and ownership checks upstream.",
    },
    "parsed_cv_sections_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training",
        "path": None,
        "format": "table-or-jsonl",
        "available_in_repo": False,
        "primary_key": ["cv_id", "section_name"],
        "required_columns": ["cv_id", "section_name", "section_text", "confidence", "parser_version"],
        "safe_training_use": "Use section text and parser confidence for ATS and summary evidence only.",
    },
    "candidate_job_sets_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "backend-api",
        "path": None,
        "format": "jsonl",
        "available_in_repo": False,
        "primary_key": ["candidate_set_id", "job_id"],
        "required_columns": ["request_id", "candidate_set_id", "profile_id", "job_id", "candidate_rank", "retrieval_source", "created_at"],
        "safe_training_use": "Use only backend-provided candidate IDs for ranking experiments; backend remains owner of job detail hydration and availability.",
    },
    "manual_labels_v1": {
        "schema_version": SCHEMA_VERSION,
        "owner": "model-training-reviewers",
        "path": None,
        "format": "table-or-jsonl",
        "available_in_repo": False,
        "primary_key": ["label_id"],
        "required_columns": ["label_id", "entity_type", "entity_id", "reviewer_id", "label_version", "score", "evidence_notes", "created_at"],
        "score_range": SCORE_RANGE_0_100,
        "safe_training_use": "Evaluation-only until governance explicitly approves another use. Do not leak reviewer labels into model input features.",
    },
}

available_paths = [REPO_ROOT / cfg["path"] for cfg in source_schema_registry.values() if cfg.get("available_in_repo") and cfg.get("path")]
missing_available_paths = [rel(path) for path in available_paths if not path.exists()]
if missing_available_paths:
    raise FileNotFoundError(f"Missing available source files: {missing_available_paths}")

source_schema_registry


{'jobs_v1': {'schema_version': 'data-snapshot-contract-v1',
  'owner': 'model-training',
  'path': 'legacy/dataset/indotech_job_cleaned.csv',
  'format': 'csv',
  'available_in_repo': True,
  'primary_key': ['job_id'],
  'required_columns': ['job_id',
   'title',
   'description',
   'requirements_concat',
   'skills_clean',
   'experience_level',
   'language_signal',
   'status'],
  'optional_columns': ['company_name',
   'normalized_title',
   'category',
   'work_type',
   'employment_type',
   'province',
   'city',
   'salary_min',
   'salary_max',
   'salary_currency',
   'source_posted_at'],
  'language_column': 'language_signal',
  'safe_training_use': 'Use job text, normalized role, skills, requirements, experience level, language, and coarse location/work-type features. Do not train on company identity as a primary fit signal.'},
 'profiles_v1': {'schema_version': 'data-snapshot-contract-v1',
  'owner': 'model-training',
  'path': 'legacy/dataset/techtalent_profile_cleaned.c

## Step 13.2 — Dataset snapshot manifests

### Purpose
Create deterministic manifests for available source files so later labels, features, pairs, and model cards can point to immutable input evidence.

### Required input
Available source files from the source schema registry.

### Action
Load each available dataset, compute row counts, column profiles, null rates, duplicate identifier counts, language distribution when present, score range when present, file size, and SHA-256 hash.

### Expected output
`reports/phase_13_snapshot_manifests.json` with one manifest per available training input file.

### Verification
Every available training input has a hash, row count, schema version, owner, required-column status, and deterministic file profile.


In [ ]:
def load_dataset(path: Path, fmt: str) -> pd.DataFrame:
    if fmt == "csv":
        return pd.read_csv(path)
    if fmt == "parquet":
        return pd.read_parquet(path, engine="pyarrow")
    if fmt == "json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported snapshot format: {fmt}")


def column_profile(df: pd.DataFrame) -> dict[str, dict[str, Any]]:
    profiles: dict[str, dict[str, Any]] = {}
    row_count = len(df)
    for column in df.columns:
        series = df[column]
        non_null = series.dropna()
        example = None if non_null.empty else non_null.iloc[0]
        item: dict[str, Any] = {
            "dtype": str(series.dtype),
            "null_count": int(series.isna().sum()),
            "null_rate": float(series.isna().mean()) if row_count else 0.0,
            "unique_count": int(series.nunique(dropna=True)),
            "example": json_safe(example),
        }
        if pd.api.types.is_numeric_dtype(series):
            item.update({
                "min": json_safe(series.min(skipna=True)),
                "max": json_safe(series.max(skipna=True)),
                "mean": json_safe(series.mean(skipna=True)),
            })
        profiles[column] = item
    return profiles


def language_distribution(df: pd.DataFrame, language_column: str | None) -> dict[str, int]:
    if not language_column or language_column not in df.columns:
        return {"UNKNOWN": int(len(df))}
    normalized = df[language_column].fillna("UNKNOWN").astype(str).str.upper().str.strip()
    normalized = normalized.where(normalized.isin(ALLOWED_LANGUAGES), "UNSUPPORTED")
    return {str(k): int(v) for k, v in normalized.value_counts(dropna=False).sort_index().items()}


def duplicate_count(df: pd.DataFrame, primary_key: list[str]) -> int | None:
    missing = [column for column in primary_key if column not in df.columns]
    if missing:
        return None
    return int(df.duplicated(subset=primary_key).sum())


def score_profile(df: pd.DataFrame, score_column: str | None) -> dict[str, Any] | None:
    if not score_column or score_column not in df.columns:
        return None
    series = pd.to_numeric(df[score_column], errors="coerce")
    return {
        "column": score_column,
        "null_or_non_numeric_count": int(series.isna().sum()),
        "min": json_safe(series.min(skipna=True)),
        "max": json_safe(series.max(skipna=True)),
        "mean": json_safe(series.mean(skipna=True)),
    }

snapshot_manifests: dict[str, dict[str, Any]] = {}
loaded_datasets: dict[str, pd.DataFrame] = {}

for dataset_name, cfg in source_schema_registry.items():
    if not cfg.get("available_in_repo") or not cfg.get("path"):
        continue
    path = REPO_ROOT / cfg["path"]
    df = load_dataset(path, cfg["format"])
    loaded_datasets[dataset_name] = df
    required_columns = cfg.get("required_columns", [])
    missing_required_columns = [column for column in required_columns if column not in df.columns]
    manifest = {
        "dataset_name": dataset_name,
        "schema_version": cfg["schema_version"],
        "owner": cfg["owner"],
        "path": rel(path),
        "format": cfg["format"],
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "columns": list(df.columns),
        "required_columns": required_columns,
        "missing_required_columns": missing_required_columns,
        "primary_key": cfg.get("primary_key", []),
        "duplicate_primary_key_count": duplicate_count(df, cfg.get("primary_key", [])),
        "language_distribution": language_distribution(df, cfg.get("language_column")),
        "score_profile": score_profile(df, cfg.get("score_column")),
        "column_profile": column_profile(df),
    }
    snapshot_manifests[dataset_name] = json_safe(manifest)

snapshot_report_path = REPORTS_DIR / "phase_13_snapshot_manifests.json"
snapshot_report_path.write_text(json.dumps(snapshot_manifests, indent=2, sort_keys=True) + "\n")

snapshot_manifests


{'jobs_v1': {'dataset_name': 'jobs_v1',
  'schema_version': 'data-snapshot-contract-v1',
  'owner': 'model-training',
  'path': 'legacy/dataset/indotech_job_cleaned.csv',
  'format': 'csv',
  'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565',
  'size_bytes': 2059443,
  'row_count': 2073,
  'column_count': 35,
  'columns': ['job_id',
   'company_name',
   'title',
   'normalized_title',
   'category',
   'description',
   'requirement_summary',
   'work_type',
   'employment_type',
   'experience_level',
   'province',
   'city',
   'salary_min',
   'salary_max',
   'salary_currency',
   'salary_display',
   'skills_top_10_names',
   'requirements_concat',
   'language_signal',
   'source_posted_at',
   'status',
   'fit_input_quality_score',
   'fit_input_has_requirements',
   'fit_input_has_skills',
   'skills_count_total',
   'description_length_chars',
   'req_len',
   '_is_tech_cat',
   '_is_tech_title',
   '_is_tech_skills',
   'is_tech',
   'tech_signal

## Step 13.3 — Model-owned output schema freezing

### Purpose
Freeze model-core output schemas before implementation so later training notebooks cannot silently emit backend-owned, wrapper-owned, unsafe, or unsupported fields.

### Required input
Model/training boundaries, Phase 12 shared config, and the Backend API OpenAPI contract.

### Action
Define output schemas for `jobFitAlignment`, `atsFriendliness`, `overallImpression`, and candidate recommendation ranking outputs. Mark blocked wrapper/backend-owned fields explicitly.

### Expected output
`reports/phase_13_model_core_schema_contracts.json` containing the model-core schema contracts and blocked field policy.

### Verification
Schema validation checks reject unsafe fields and confirm model-owned outputs stay within API wrapper boundaries.


In [ ]:
OPENAPI_PATH = REPO_ROOT / "references/docs/generated/openapi.json"
openapi_contract = json.loads(OPENAPI_PATH.read_text())

MODEL_CORE_OUTPUT_SCHEMAS: dict[str, dict[str, Any]] = {
    "jobFitAlignment": {
        "schema_version": "model-core-job-fit-alignment-v1",
        "owner": "model-core",
        "type": "object",
        "required": ["score", "summarySignals", "matchedSkills", "missingSkills", "confidenceNotes"],
        "properties": {
            "score": {"type": "integer", "minimum": 0, "maximum": 100},
            "summarySignals": {"type": "array", "items": {"type": "string"}},
            "missingSignals": {"type": "array", "items": {"type": "string"}},
            "matchedSkills": {"type": "array", "items": {"type": "string"}},
            "missingSkills": {"type": "array", "items": {"type": "string"}},
            "confidenceNotes": {"type": "array", "items": {"type": "string"}},
        },
        "blocked_fields": ["topActionables", "sectionReviews", "jobDetails", "hydratedJobDetails", "auth", "persistence"],
    },
    "atsFriendliness": {
        "schema_version": "model-core-ats-friendliness-v1",
        "owner": "model-core",
        "type": "object",
        "required": ["score", "detectedIssues", "confidenceNotes"],
        "properties": {
            "score": {"type": "integer", "minimum": 0, "maximum": 100},
            "detectedIssues": {"type": "array", "items": {"type": "string"}},
            "confidenceNotes": {"type": "array", "items": {"type": "string"}},
        },
        "blocked_fields": ["rewrittenCv", "sectionReviews", "topActionables", "storageUrl", "rawFileBytes"],
    },
    "overallImpression": {
        "schema_version": "model-core-overall-impression-v1",
        "owner": "model-core",
        "type": "object",
        "required": ["score", "summary", "evidenceKeys", "confidenceNotes"],
        "properties": {
            "score": {"type": "integer", "minimum": 0, "maximum": 100},
            "summary": {"type": "string"},
            "evidenceKeys": {"type": "array", "items": {"type": "string"}},
            "confidenceNotes": {"type": "array", "items": {"type": "string"}},
        },
        "blocked_fields": ["topActionables", "sectionReviews", "hiringDecision", "salaryClaim", "unsupportedSeniorityClaim"],
    },
    "recommendationRanking": {
        "schema_version": "model-core-recommendation-ranking-v1",
        "owner": "model-core",
        "type": "object",
        "required": ["jobId", "matchScore", "matchLevel", "matchedSkills", "missingSkills", "rankingSignals"],
        "properties": {
            "jobId": {"type": "string"},
            "matchScore": {"type": "integer", "minimum": 0, "maximum": 100},
            "matchLevel": {"type": "string", "enum": ["LOW", "MEDIUM", "HIGH", "UNKNOWN"]},
            "matchedSkills": {"type": "array", "items": {"type": "string"}},
            "missingSkills": {"type": "array", "items": {"type": "string"}},
            "rankingSignals": {"type": "array", "items": {"type": "string"}},
        },
        "blocked_fields": ["title", "company", "companyName", "description", "location", "salary", "visibility", "availability", "hydratedJobDetails"],
    },
}

WRAPPER_OR_BACKEND_OWNED_FIELDS = sorted({
    "topActionables",
    "sectionReviews",
    "hydratedJobDetails",
    "jobDetails",
    "auth",
    "persistence",
    "requestValidation",
    "OpenAIWrapperCopy",
    "title",
    "company",
    "companyName",
    "visibility",
    "availability",
    "storageSignedUrl",
    "rawFileBytes",
})

schema_contract_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "openapi_reference": {
        "path": rel(OPENAPI_PATH),
        "sha256": sha256_file(OPENAPI_PATH),
        "title": openapi_contract.get("info", {}).get("title"),
        "version": openapi_contract.get("info", {}).get("version"),
    },
    "model_core_output_schemas": MODEL_CORE_OUTPUT_SCHEMAS,
    "wrapper_or_backend_owned_fields": WRAPPER_OR_BACKEND_OWNED_FIELDS,
    "boundary_policy": "Model-core notebooks may emit only model-owned signals. Backend/API wrapper owns auth, persistence, hydration, product copy assembly, topActionables, and sectionReviews.",
}

schema_contract_path = REPORTS_DIR / "phase_13_model_core_schema_contracts.json"
schema_contract_path.write_text(json.dumps(json_safe(schema_contract_report), indent=2, sort_keys=True) + "\n")

schema_contract_report


{'phase_id': 'phase_13_data_snapshot_contract_freezing',
 'schema_version': 'data-snapshot-contract-v1',
 'generated_at': '2026-06-02T04:44:56.398970+00:00',
 'openapi_reference': {'path': 'references/docs/generated/openapi.json',
  'sha256': 'aca7181f441e5fc969322795c8db78fbdaeab6e6427e89fa0608025f525bc68e',
  'title': 'Bisakerja Backend API',
  'version': '0.1.0'},
 'model_core_output_schemas': {'jobFitAlignment': {'schema_version': 'model-core-job-fit-alignment-v1',
   'owner': 'model-core',
   'type': 'object',
   'required': ['score',
    'summarySignals',
    'matchedSkills',
    'missingSkills',
    'confidenceNotes'],
   'properties': {'score': {'type': 'integer', 'minimum': 0, 'maximum': 100},
    'summarySignals': {'type': 'array', 'items': {'type': 'string'}},
    'missingSignals': {'type': 'array', 'items': {'type': 'string'}},
    'matchedSkills': {'type': 'array', 'items': {'type': 'string'}},
    'missingSkills': {'type': 'array', 'items': {'type': 'string'}},
    'confi

## Step 13.4 — Validation gates

### Purpose
Fail early when source data or output schemas violate frozen contracts.

### Required input
Snapshot manifests, loaded datasets, allowed languages, score ranges, and blocked field policy.

### Action
Validate missing IDs, duplicate IDs, unsupported languages, invalid weak-label scores, missing required columns, missing file hashes, and unsafe model-core output fields.

### Expected output
`reports/phase_13_data_snapshot_contract_freezing.json` with pass/fail validation details and acceptance criteria status.

### Verification
A clean run must pass with current repository data. Injecting missing IDs, duplicate primary keys, unsupported language values, invalid scores, or blocked output fields must produce validation failures.


In [ ]:
validation_checks: list[dict[str, Any]] = []


def add_check(name: str, passed: bool, details: Any) -> None:
    validation_checks.append({"name": name, "passed": bool(passed), "details": json_safe(details)})

for dataset_name, cfg in source_schema_registry.items():
    if not cfg.get("available_in_repo"):
        continue
    df = loaded_datasets[dataset_name]
    manifest = snapshot_manifests[dataset_name]
    add_check(
        f"{dataset_name}: required columns present",
        not manifest["missing_required_columns"],
        {"missing_required_columns": manifest["missing_required_columns"]},
    )
    add_check(
        f"{dataset_name}: snapshot hash present",
        bool(manifest.get("sha256")) and len(manifest.get("sha256", "")) == 64,
        {"sha256": manifest.get("sha256")},
    )
    add_check(
        f"{dataset_name}: row count positive",
        manifest["row_count"] > 0,
        {"row_count": manifest["row_count"]},
    )
    primary_key = cfg.get("primary_key", [])
    if primary_key and all(column in df.columns for column in primary_key):
        missing_id_rows = int(df[primary_key].isna().any(axis=1).sum())
        add_check(f"{dataset_name}: primary key has no missing values", missing_id_rows == 0, {"missing_id_rows": missing_id_rows})
        duplicate_rows = int(df.duplicated(subset=primary_key).sum())
        add_check(f"{dataset_name}: primary key has no duplicates", duplicate_rows == 0, {"duplicate_rows": duplicate_rows})
    language_column = cfg.get("language_column")
    if language_column and language_column in df.columns:
        normalized_languages = df[language_column].fillna("UNKNOWN").astype(str).str.upper().str.strip()
        unsupported_languages = sorted(set(normalized_languages) - ALLOWED_LANGUAGES)
        add_check(f"{dataset_name}: supported language values", not unsupported_languages, {"unsupported_languages": unsupported_languages})
    score_column = cfg.get("score_column")
    score_range = cfg.get("score_range")
    if score_column and score_column in df.columns and score_range:
        scores = pd.to_numeric(df[score_column], errors="coerce")
        invalid_scores = int((scores.isna() | (scores < score_range["min"]) | (scores > score_range["max"])).sum())
        add_check(f"{dataset_name}: score range valid", invalid_scores == 0, {"invalid_scores": invalid_scores, "score_range": score_range})

for output_name, schema in MODEL_CORE_OUTPUT_SCHEMAS.items():
    properties = set(schema.get("properties", {}).keys())
    blocked = set(schema.get("blocked_fields", [])) | set(WRAPPER_OR_BACKEND_OWNED_FIELDS)
    unsafe_overlap = sorted(properties & blocked)
    add_check(f"{output_name}: no blocked fields in properties", not unsafe_overlap, {"unsafe_overlap": unsafe_overlap})
    required = set(schema.get("required", []))
    missing_required_properties = sorted(required - properties)
    add_check(f"{output_name}: required properties defined", not missing_required_properties, {"missing_required_properties": missing_required_properties})

acceptance = {
    "every_training_input_file_has_hash_row_count_schema_version_owner": all(
        manifest.get("sha256") and manifest.get("row_count", 0) > 0 and manifest.get("schema_version") and manifest.get("owner")
        for manifest in snapshot_manifests.values()
    ),
    "dataset_snapshot_regenerates_from_clean_kernel": all(check["passed"] for check in validation_checks),
    "model_core_schema_aligned_with_openapi_boundaries": all(
        not (set(schema.get("properties", {})) & set(schema.get("blocked_fields", []))) for schema in MODEL_CORE_OUTPUT_SCHEMAS.values()
    ),
    "unsafe_wrapper_backend_fields_blocked": bool(WRAPPER_OR_BACKEND_OWNED_FIELDS) and all(
        not (set(schema.get("properties", {})) & set(WRAPPER_OR_BACKEND_OWNED_FIELDS)) for schema in MODEL_CORE_OUTPUT_SCHEMAS.values()
    ),
}

phase_report = {
    "phase_id": PHASE_ID,
    "status": "complete" if all(acceptance.values()) else "blocked",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "runtime": {"python": platform.python_version(), "platform": platform.platform(), "pandas": pd.__version__, "seed": SEED},
    "source_schema_registry": source_schema_registry,
    "snapshot_manifest_report": {"path": rel(snapshot_report_path), "sha256": sha256_file(snapshot_report_path)},
    "schema_contract_report": {"path": rel(schema_contract_path), "sha256": sha256_file(schema_contract_path)},
    "validation_checks": validation_checks,
    "acceptance": acceptance,
}

phase_report_path = REPORTS_DIR / "phase_13_data_snapshot_contract_freezing.json"
phase_report_path.write_text(json.dumps(json_safe(phase_report), indent=2, sort_keys=True) + "\n")

if not all(check["passed"] for check in validation_checks):
    failed = [check for check in validation_checks if not check["passed"]]
    raise AssertionError(f"Phase 13 validation failed: {failed}")

if not all(acceptance.values()):
    raise AssertionError(f"Phase 13 acceptance failed: {acceptance}")

phase_report


{'phase_id': 'phase_13_data_snapshot_contract_freezing',
 'status': 'complete',
 'generated_at': '2026-06-02T04:44:56.423354+00:00',
 'runtime': {'python': '3.14.2',
  'platform': 'macOS-14.8.2-arm64-arm-64bit-Mach-O',
  'pandas': '3.0.3',
  'seed': 202613},
 'source_schema_registry': {'jobs_v1': {'schema_version': 'data-snapshot-contract-v1',
   'owner': 'model-training',
   'path': 'legacy/dataset/indotech_job_cleaned.csv',
   'format': 'csv',
   'available_in_repo': True,
   'primary_key': ['job_id'],
   'required_columns': ['job_id',
    'title',
    'description',
    'requirements_concat',
    'skills_clean',
    'experience_level',
    'language_signal',
    'status'],
   'optional_columns': ['company_name',
    'normalized_title',
    'category',
    'work_type',
    'employment_type',
    'province',
    'city',
    'salary_min',
    'salary_max',
    'salary_currency',
    'source_posted_at'],
   'language_column': 'language_signal',
   'safe_training_use': 'Use job text, nor

## Step 13.5 — Snapshot report references for model card

### Purpose
Make the frozen data snapshot and schema contracts discoverable from later model cards and artifact manifests.

### Required input
Snapshot manifest report, schema contract report, Phase 13 validation report, and model-card template requirements from earlier phases.

### Action
Write a model-card template fragment that references frozen snapshot and schema reports with SHA-256 hashes.

### Expected output
`reports/model_card_jobfit_v2_template.json` contains data snapshot references that later export notebooks can copy into release model cards.

### Verification
The template references existing report files and their hashes. Later model cards must include these references before production-readiness claims.


In [ ]:
model_card_template = {
    "model_name": "jobfit-v2",
    "model_version": "jobfit-v2-notebook-bootstrap",
    "schema_version": "model-card-template-v1",
    "intended_use": "Notebook-first job-fit scoring experiments after frozen data snapshots, label governance, baseline evaluation, and contract validation pass.",
    "blocked_use": [
        "Production hiring decision automation",
        "User-facing production score claims without calibration evidence",
        "Backend-owned hydration or OpenAI wrapper copy generation",
    ],
    "data_snapshot_references": {
        "snapshot_manifests": {"path": rel(snapshot_report_path), "sha256": sha256_file(snapshot_report_path)},
        "model_core_schema_contracts": {"path": rel(schema_contract_path), "sha256": sha256_file(schema_contract_path)},
        "phase_13_validation_report": {"path": rel(phase_report_path), "note": "Final report hash is recorded by artifact manifests after report write completes."},
    },
    "required_before_promotion": [
        "pairs_v2 manifest from Phase 15",
        "human validation label manifest from Phase 16",
        "baseline v2 metrics from Phase 17",
        "calibration and export manifest from Phase 22",
        "contract validation report from Phase 23",
        "final gate report from Phase 24",
    ],
}

model_card_template_path = REPORTS_DIR / "model_card_jobfit_v2_template.json"
model_card_template_path.write_text(json.dumps(json_safe(model_card_template), indent=2, sort_keys=True) + "\n")

# Refresh Phase 13 report with final template reference.
phase_report["model_card_template"] = {"path": rel(model_card_template_path), "sha256": sha256_file(model_card_template_path)}
phase_report_path.write_text(json.dumps(json_safe(phase_report), indent=2, sort_keys=True) + "\n")

model_card_template


{'model_name': 'jobfit-v2',
 'model_version': 'jobfit-v2-notebook-bootstrap',
 'schema_version': 'model-card-template-v1',
 'intended_use': 'Notebook-first job-fit scoring experiments after frozen data snapshots, label governance, baseline evaluation, and contract validation pass.',
 'blocked_use': ['Production hiring decision automation',
  'User-facing production score claims without calibration evidence',
  'Backend-owned hydration or OpenAI wrapper copy generation'],
 'data_snapshot_references': {'snapshot_manifests': {'path': 'reports/phase_13_snapshot_manifests.json',
   'sha256': '6ce9dad49415cb95efd29bb3d18e31dd4b9f609dc9c78f4ef3285fa08ff63153'},
  'model_core_schema_contracts': {'path': 'reports/phase_13_model_core_schema_contracts.json',
   'sha256': '74d284763860704a5b8d82cf95d00eaae8ecf50a85668ab47ab058ebf638350b'},
  'phase_13_validation_report': {'path': 'reports/phase_13_data_snapshot_contract_freezing.json',
   'note': 'Final report hash is recorded by artifact manifest